# Base analítica de dipirona para o TCC de Farmácia

Versão auditável para Isabel Giovana Fetter Schuch (UFCSPA). Estudo descritivo e exploratório. O notebook original [coleta preliminar](https://github.com/mlab-inf-ufrgs/vigimed/tree/9d04d156a3a60ed3ef83ff1d0c01335102a462b3/eda/dipirona) permanece preservado. O CSV detalhado usa uma linha por notificação × composição suspeita × assinatura de reação; linhas não equivalem a notificações.

## 1. Configuração e dependências

DuckDB e pandas já disponíveis no ambiente. O limiar de raras e a referência temporal são parâmetros explícitos. DATA_NOTIFICACAO não recebe fallback. Usar outro diretório de saída para análises por inclusão.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/03_gold").is_dir())
sys.path.insert(0, str(ROOT / "ad-hoc-analysis/dipirona"))
from base_analitica import Study
LIMIAR_ASSOCIACOES = 50
CAMPO_DATA = "DATA_NOTIFICACAO"
OUTPUT = ROOT / "ad-hoc-analysis/dipirona/output"
study = Study(ROOT, threshold=LIMIAR_ASSOCIACOES, date_field=CAMPO_DATA, output=OUTPUT)

## 2. Leitura dos Parquet Gold

As cinco tabelas são registradas como views, sem recriar Bronze/Silver/Gold. As chaves e o snapshot corrente são verificados.

In [ ]:
study.load()
[(t, study.scalar(f"select count(*) from {t}")) for t in ("dim_notificacoes", "fat_medicamentos", "fat_reacoes", "dim_soc_llt", "dim_atc")]

## 3. Diagnóstico das fontes e cardinalidades

Inspeciona esquemas reais, datas, papéis e reproduz a seleção antiga antes do DISTINCT final. A categoria Outros antiga não exige dipirona nem papel suspeito. A junção bruta multiplica produtos por reações.

In [ ]:
study.diagnose()
for report in ("fontes", "datas", "papeis", "legado"):
    print(report)
    display(study.reports[report])

## 4. Identificação da dipirona e associações

Busca candidatos em princípios ativos, nome WHO Drug, ATC nível 4/5 e descrição ATC. O código ATC sozinho não confirma composição.

In [ ]:
study.identify()
study.df("select evidencia_textual,count(*) registros from candidatos group by 1")

## 5. Normalização dos princípios ativos

Normaliza caixa, acentos, espaços e separadores. Aliases PT/EN ficam explícitos no módulo. Sal magnésico e escopolamina sem sal informado permanecem distinguíveis. Composição desconhecida exige revisão.

In [ ]:
study.normalize_actives()
display(study.reports["mapa_composicoes"])
display(study.reports["papeis_dipirona"])

## 6. Classificação dos grupos de medicamentos

Apenas SUSPEITO e datas dentro do período. Frequências usam notificações distintas por composição. Raras abaixo do limiar entram em Outras associações; dipirona isolada é mantida separada.

In [ ]:
study.classify_groups()
study.reports["frequencias_associacao"]

## 7. Construção da base analítica

Agrega produtos de mesma composição antes de unir às reações. Canoniza o SOC primário por LLT. Reações com atributos clínicos distintos são preservadas; assinaturas indistinguíveis são agregadas com multiplicidade e hashes originais. Sem ligação produto–reação específica, não há atribuição causal.

In [ ]:
study.build()
study.df("select id_notificacao,ano_notificacao,sexo,grupo_medicamento,grave,pt_name,id_reacao from analitica order by id_notificacao limit 10")

## 8. Verificações de qualidade

Assertions sobre dados reais: datas, evidência, papéis, ATC, exclusividade, limiar, grão, cardinalidade e MedDRA. Os testes sintéticos exercitam casos de fronteira e o build real.

In [ ]:
study.validate()
import unittest
suite = unittest.defaultTestLoader.discover(str(ROOT / "ad-hoc-analysis/dipirona"), pattern="test_*.py")
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()
study.df("select count(distinct id_notificacao) notificacoes,count(distinct id_reacao) reacoes,count(*) linhas from analitica")

## 9. Resumos descritivos

Os resumos derivam do mesmo universo da base. Totais por ano são aditivos. Notificações por grupo e por PT/SOC podem se sobrepor. A sensibilidade temporal mostra o efeito de trocar a referência; datas ausentes permanecem ausentes.

In [ ]:
study.summarize()
for report in ("fluxo", "sensibilidade_datas", "orfaos", "multiplicidades", "resumo_notificacoes_ano", "resumo_medicamentos"):
    print(report)
    display(study.reports[report])

## 10. Exportação do CSV

CSV UTF-8 com BOM, separador ;, sem índice e datas ISO. A exportação relê e compara todas as células. Dicionário, auditorias, resumos, manifesto SHA-256 e relatório de QA são gravados no mesmo diretório. No Excel, usar Contagem Distinta de id_notificacao ou id_reacao, excluindo IDs de reação vazios.

In [ ]:
csv_principal = study.export()
print(csv_principal)
print(f"{len(study.checks)} verificações sobre dados reais aprovadas.")
display(study.reports["testes"])
print((OUTPUT / "qa_relatorio.md").read_text(encoding="utf-8")[:1800])